# Transformers from Scratch: See Every Part Working

This notebook builds a complete Transformer (encoder + decoder) step by step, the same architecture as the 2017 paper *Attention Is All You Need*.

**What you will do**
1. Turn text into tokens and embeddings
2. Add positional encoding
3. Compute self-attention by hand and see the attention scores
4. Build multi-head attention, masking, feed-forward, add & norm
5. Assemble the full encoder and decoder
6. **Train the Transformer** on a small task (reversing a sequence of numbers)
7. Look inside the trained model and watch attention at work
8. Compare with real pre-trained models (BERT, GPT-2) using Hugging Face

**How to run:** open in Google Colab, then *Runtime → Run all*. Parts 1–10 run on a normal CPU in about 1–2 minutes. Part 11 needs internet to download models.

In [ ]:
import math
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt

torch.manual_seed(42)
print("PyTorch version:", torch.__version__)

def show_heatmap(matrix, xlabels, ylabels, title, xlabel="Key (looked at)", ylabel="Query (looking)"):
    matrix = matrix.detach().cpu().numpy() if torch.is_tensor(matrix) else matrix
    fig, ax = plt.subplots(figsize=(max(4, len(xlabels) * 0.6), max(3, len(ylabels) * 0.5)))
    im = ax.imshow(matrix, cmap="viridis")
    ax.set_xticks(range(len(xlabels))); ax.set_xticklabels(xlabels, rotation=45, ha="right")
    ax.set_yticks(range(len(ylabels))); ax.set_yticklabels(ylabels)
    ax.set_xlabel(xlabel); ax.set_ylabel(ylabel); ax.set_title(title)
    fig.colorbar(im, ax=ax, fraction=0.046)
    plt.tight_layout(); plt.show()

## Part 1: Tokenization

A model cannot read text. First we split the sentence into **tokens** and give each token a number (**token ID**).

Real models split into word pieces (e.g. "tired" → "tir", "ed"). Here we keep it simple and split on spaces.

In [ ]:
sentence = "the cat sat on the mat because it was tired"
tokens = sentence.split()

vocab = {word: idx for idx, word in enumerate(sorted(set(tokens)))}
token_ids = torch.tensor([vocab[t] for t in tokens])

print("Tokens:   ", tokens)
print("Vocabulary:", vocab)
print("Token IDs:", token_ids.tolist())

## Part 2: Embeddings

Each token ID becomes a **vector** (a list of numbers). This vector represents the word's meaning.

`d_model` is the vector size. Real models use 768–12,288. We use 8 so we can print it.

Right now the numbers are random. **Training** is what makes similar words get similar vectors.

In [ ]:
d_model = 8
embedding = nn.Embedding(num_embeddings=len(vocab), embedding_dim=d_model)

x = embedding(token_ids)            # shape: (number of tokens, d_model)
print("Embedding matrix shape:", x.shape)
print("\nVector for 'cat':")
print(x[1].detach().numpy().round(3))

## Part 3: Positional Encoding

A Transformer looks at all words **at the same time**, so it has no idea of word order. "Dog bites man" and "man bites dog" would look identical.

The fix: add a unique **position pattern** to each word's vector, built from sine and cosine waves of different speeds.

$$PE_{(pos, 2i)} = \sin\left(\frac{pos}{10000^{2i/d}}\right) \qquad PE_{(pos, 2i+1)} = \cos\left(\frac{pos}{10000^{2i/d}}\right)$$

In [ ]:
def positional_encoding(max_len, d_model):
    pe = torch.zeros(max_len, d_model)
    position = torch.arange(0, max_len).unsqueeze(1).float()
    div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))
    pe[:, 0::2] = torch.sin(position * div_term)
    pe[:, 1::2] = torch.cos(position * div_term)
    return pe

# Picture with a bigger size so the wave pattern is visible
pe_big = positional_encoding(max_len=50, d_model=64)
plt.figure(figsize=(9, 4))
plt.imshow(pe_big.numpy(), cmap="RdBu", aspect="auto")
plt.xlabel("Embedding dimension"); plt.ylabel("Position in sentence")
plt.title("Positional encoding: every row (position) has a unique pattern")
plt.colorbar(); plt.show()

# Add position information to our sentence
pe = positional_encoding(len(tokens), d_model)
x = embedding(token_ids) + pe
print("Input to the Transformer (embedding + position):", x.shape)

## Part 4: Self-Attention, Step by Step

This is the heart of the Transformer. Each word creates three vectors:

| Vector | Question it answers |
|---|---|
| **Query (Q)** | What am I looking for? |
| **Key (K)** | What do I contain? |
| **Value (V)** | What information do I pass on? |

Then:
1. Compare every Query with every Key → **scores**
2. Divide by √d (keeps numbers stable)
3. **Softmax** → turns scores into percentages that add up to 1 (the attention weights)
4. Use the weights to mix the Values

$$\text{Attention}(Q, K, V) = \text{softmax}\left(\frac{QK^T}{\sqrt{d_k}}\right)V$$

In [ ]:
W_q = nn.Linear(d_model, d_model, bias=False)
W_k = nn.Linear(d_model, d_model, bias=False)
W_v = nn.Linear(d_model, d_model, bias=False)

Q = W_q(x)
K = W_k(x)
V = W_v(x)
print("Q, K, V shapes:", Q.shape, K.shape, V.shape)

# Step 1 + 2: scores
scores = Q @ K.T / math.sqrt(d_model)
print("\nScores shape (word x word):", scores.shape)

# Step 3: softmax -> attention weights
weights = F.softmax(scores, dim=-1)
print("Each row sums to 1:", weights.sum(dim=-1).detach().numpy().round(2))

# Step 4: mix the values
output = weights @ V
print("Output shape:", output.shape, "(same shape as input, but now every word carries context)")

In [ ]:
show_heatmap(weights, tokens, tokens, "Self-attention weights (random, untrained model)")

it_idx = tokens.index("it")
print("How much 'it' attends to each word:")
for word, w in zip(tokens, weights[it_idx]):
    print(f"  {word:>8}: {w.item():.2f}  " + "#" * int(w.item() * 50))

> **Teaching point:** the pattern above is random because the weights `W_q`, `W_k`, `W_v` are random. During training, the model learns these weights so that "it" pays strong attention to "cat". In Part 10 you will see a *trained* model's attention, which is clearly structured.

Now let's package attention into a reusable function, with optional **masking** (used in Part 6).

In [ ]:
def scaled_dot_product_attention(Q, K, V, mask=None):
    d_k = Q.size(-1)
    scores = Q @ K.transpose(-2, -1) / math.sqrt(d_k)
    if mask is not None:
        scores = scores.masked_fill(mask == 0, float("-inf"))   # blocked positions get 0% after softmax
    weights = F.softmax(scores, dim=-1)
    return weights @ V, weights

## Part 5: Multi-Head Attention

One attention calculation can only focus on one kind of relationship. **Multi-head attention** runs several attention "heads" in parallel, each with its own Q, K, V weights.

One head might learn grammar, another might learn which noun a pronoun refers to. Their outputs are joined and mixed by a final linear layer.

In [ ]:
class MultiHeadAttention(nn.Module):
    def __init__(self, d_model, num_heads):
        super().__init__()
        assert d_model % num_heads == 0
        self.num_heads = num_heads
        self.d_k = d_model // num_heads
        self.W_q = nn.Linear(d_model, d_model)
        self.W_k = nn.Linear(d_model, d_model)
        self.W_v = nn.Linear(d_model, d_model)
        self.W_o = nn.Linear(d_model, d_model)
        self.attn_weights = None          # saved so we can look at it later

    def split_heads(self, x):
        # (batch, seq, d_model) -> (batch, heads, seq, d_k)
        b, s, _ = x.shape
        return x.view(b, s, self.num_heads, self.d_k).transpose(1, 2)

    def forward(self, query, key, value, mask=None):
        Q = self.split_heads(self.W_q(query))
        K = self.split_heads(self.W_k(key))
        V = self.split_heads(self.W_v(value))
        out, weights = scaled_dot_product_attention(Q, K, V, mask)
        self.attn_weights = weights.detach()
        b, h, s, d = out.shape
        out = out.transpose(1, 2).contiguous().view(b, s, h * d)   # join the heads back
        return self.W_o(out)

mha = MultiHeadAttention(d_model=8, num_heads=2)
batch = x.unsqueeze(0)                  # add a batch dimension: (1, 10, 8)
out = mha(batch, batch, batch)          # self-attention: Q, K, V all come from the same sentence
print("Input :", batch.shape)
print("Output:", out.shape)
print("Attention weights per head:", mha.attn_weights.shape, "-> (batch, heads, query, key)")

## Part 6: Masking (used by the Decoder)

When a decoder **generates** text, it writes one word at a time. During training it must not "peek" at future words, or it would just copy the answer.

A **causal mask** blocks every position from seeing positions after it. The grid below shows what each word is allowed to see (yellow = allowed).

In [ ]:
def causal_mask(size):
    return torch.tril(torch.ones(size, size)).bool()

mask = causal_mask(len(tokens))
show_heatmap(mask.float(), tokens, tokens, "Causal mask: each word can only see itself and earlier words")

_, masked_weights = scaled_dot_product_attention(Q, K, V, mask)
show_heatmap(masked_weights, tokens, tokens, "Attention weights WITH the mask (upper triangle is zero)")

## Part 7: Feed-Forward Network, Add & Norm → Encoder Layer

Each Transformer layer has two more pieces:

- **Feed-forward network:** a small 2-layer neural network applied to every word separately. Attention *gathers* information from other words; feed-forward *processes* it.
- **Add & Norm:**
  - *Add (residual connection):* add the layer's input back to its output, so information is never lost in deep stacks.
  - *Norm (layer normalization):* keeps the numbers in a stable range so training works.

**Encoder layer = Self-Attention → Add & Norm → Feed-Forward → Add & Norm**

In [ ]:
class FeedForward(nn.Module):
    def __init__(self, d_model, d_ff, dropout=0.1):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(d_model, d_ff), nn.ReLU(), nn.Dropout(dropout), nn.Linear(d_ff, d_model)
        )
    def forward(self, x):
        return self.net(x)


class EncoderLayer(nn.Module):
    def __init__(self, d_model, num_heads, d_ff, dropout=0.1):
        super().__init__()
        self.self_attn = MultiHeadAttention(d_model, num_heads)
        self.ff = FeedForward(d_model, d_ff, dropout)
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.drop = nn.Dropout(dropout)

    def forward(self, x, src_mask=None):
        x = self.norm1(x + self.drop(self.self_attn(x, x, x, src_mask)))   # attention + add & norm
        x = self.norm2(x + self.drop(self.ff(x)))                           # feed-forward + add & norm
        return x

enc_layer = EncoderLayer(d_model=8, num_heads=2, d_ff=32)
print("Encoder layer output:", enc_layer(batch).shape)

## Part 8: Decoder Layer

The decoder layer has **three** sub-layers:

1. **Masked self-attention:** looks at the words generated so far (no peeking ahead)
2. **Cross-attention:** Queries come from the decoder, Keys and Values come from the **encoder output**. This is the bridge: "which part of the input should I use to write the next word?"
3. **Feed-forward**

Each one is followed by Add & Norm.

In [ ]:
class DecoderLayer(nn.Module):
    def __init__(self, d_model, num_heads, d_ff, dropout=0.1):
        super().__init__()
        self.self_attn = MultiHeadAttention(d_model, num_heads)
        self.cross_attn = MultiHeadAttention(d_model, num_heads)
        self.ff = FeedForward(d_model, d_ff, dropout)
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.norm3 = nn.LayerNorm(d_model)
        self.drop = nn.Dropout(dropout)

    def forward(self, x, enc_out, src_mask=None, tgt_mask=None):
        x = self.norm1(x + self.drop(self.self_attn(x, x, x, tgt_mask)))                # 1. masked self-attention
        x = self.norm2(x + self.drop(self.cross_attn(x, enc_out, enc_out, src_mask)))   # 2. cross-attention
        x = self.norm3(x + self.drop(self.ff(x)))                                       # 3. feed-forward
        return x

## Part 9: The Full Transformer

Now we stack everything, exactly like the architecture diagram:

```
Input tokens  → Embedding + Position → [Encoder layer] × N ─────────┐
                                                                    ↓ (cross-attention)
Output so far → Embedding + Position → [Decoder layer] × N → Linear + Softmax → next token
```

In [ ]:
class Transformer(nn.Module):
    def __init__(self, src_vocab, tgt_vocab, d_model=64, num_heads=4, num_layers=2, d_ff=128,
                 max_len=100, dropout=0.1):
        super().__init__()
        self.d_model = d_model
        self.src_embed = nn.Embedding(src_vocab, d_model)
        self.tgt_embed = nn.Embedding(tgt_vocab, d_model)
        self.register_buffer("pe", positional_encoding(max_len, d_model))
        self.encoder = nn.ModuleList([EncoderLayer(d_model, num_heads, d_ff, dropout) for _ in range(num_layers)])
        self.decoder = nn.ModuleList([DecoderLayer(d_model, num_heads, d_ff, dropout) for _ in range(num_layers)])
        self.out = nn.Linear(d_model, tgt_vocab)     # the final "Linear" layer (softmax is applied in the loss)
        self.drop = nn.Dropout(dropout)

    def embed(self, tokens, table):
        return self.drop(table(tokens) * math.sqrt(self.d_model) + self.pe[: tokens.size(1)])

    def encode(self, src, src_mask=None):
        x = self.embed(src, self.src_embed)
        for layer in self.encoder:
            x = layer(x, src_mask)
        return x

    def decode(self, tgt, enc_out, src_mask=None):
        tgt_mask = causal_mask(tgt.size(1)).to(tgt.device)
        x = self.embed(tgt, self.tgt_embed)
        for layer in self.decoder:
            x = layer(x, enc_out, src_mask, tgt_mask)
        return self.out(x)

    def forward(self, src, tgt):
        return self.decode(tgt, self.encode(src))

In [ ]:
demo = Transformer(src_vocab=20, tgt_vocab=20)
src = torch.randint(0, 20, (2, 7))   # 2 sentences, 7 tokens each
tgt = torch.randint(0, 20, (2, 5))   # 2 partial outputs, 5 tokens each

enc_out = demo.encode(src)
logits = demo.decode(tgt, enc_out)
print("Source tokens       :", src.shape, " (batch, source length)")
print("Encoder output      :", enc_out.shape, " (batch, source length, d_model)")
print("Decoder output      :", logits.shape, " (batch, target length, vocabulary size)")
print("Number of parameters:", f"{sum(p.numel() for p in demo.parameters()):,}")
print("\nFor comparison: GPT-3 has 175,000,000,000 parameters. Same idea, much bigger.")

## Part 10: Train the Transformer

An untrained Transformer outputs nonsense. Let's train ours on a small task: **reverse a sequence of numbers**.

| Input | Output |
|---|---|
| 3 7 1 9 4 | 4 9 1 7 3 |

This is a "translation" task, like English → Hindi, but simple enough to train in 1–2 minutes on a CPU.

Special tokens:
- `0` = PAD (padding), `1` = BOS (begin output), `2` = EOS (end of output)
- `3`–`12` = the digits 0–9

In [ ]:
PAD, BOS, EOS = 0, 1, 2
VOCAB = 13
SEQ_LEN = 8

def make_batch(batch_size):
    digits = torch.randint(3, VOCAB, (batch_size, SEQ_LEN))
    target = digits.flip(dims=[1])                                     # the reversed sequence
    bos = torch.full((batch_size, 1), BOS)
    eos = torch.full((batch_size, 1), EOS)
    decoder_input  = torch.cat([bos, target], dim=1)                   # BOS 4 9 1 7 3 ...
    decoder_target = torch.cat([target, eos], dim=1)                   # 4 9 1 7 3 ... EOS
    return digits, decoder_input, decoder_target

src, dec_in, dec_out = make_batch(1)
print("Encoder input  :", (src[0] - 3).tolist())
print("Decoder input  : [BOS] +", (dec_in[0, 1:] - 3).tolist())
print("Decoder target :", (dec_out[0, :-1] - 3).tolist(), "+ [EOS]")

**Teacher forcing:** during training the decoder is given the correct previous words and asked to predict the next one at every position at once. The causal mask (Part 6) makes sure it can't cheat.

In [ ]:
torch.manual_seed(0)
model = Transformer(src_vocab=VOCAB, tgt_vocab=VOCAB, d_model=64, num_heads=4, num_layers=2, d_ff=128, dropout=0.0)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=2000)   # slowly lower the learning rate
loss_fn = nn.CrossEntropyLoss()

losses = []
model.train()
for step in range(1, 2001):
    src, dec_in, dec_out = make_batch(64)
    logits = model(src, dec_in)                                    # forward pass
    loss = loss_fn(logits.reshape(-1, VOCAB), dec_out.reshape(-1)) # how wrong are we?
    optimizer.zero_grad()
    loss.backward()                                                # backpropagation
    optimizer.step()                                               # update all weights (incl. W_q, W_k, W_v)
    scheduler.step()
    losses.append(loss.item())
    if step % 250 == 0:
        print(f"step {step:4d} | loss {loss.item():.4f}")

plt.figure(figsize=(7, 3))
plt.plot(losses)
plt.xlabel("Training step"); plt.ylabel("Loss"); plt.title("Training loss (lower = better)")
plt.show()

### Generate answers one token at a time

This is exactly how ChatGPT and Claude produce text: predict the most likely next token, add it to the output, repeat until EOS. This is called **autoregressive generation**.

In [ ]:
@torch.no_grad()
def generate(model, src):
    model.eval()
    enc_out = model.encode(src)
    out = torch.full((src.size(0), 1), BOS)
    for _ in range(SEQ_LEN + 1):
        logits = model.decode(out, enc_out)
        next_token = logits[:, -1].argmax(dim=-1, keepdim=True)    # pick the most likely next token
        out = torch.cat([out, next_token], dim=1)
    return out[:, 1:]

test_src, _, test_target = make_batch(5)
predictions = generate(model, test_src)
for s, p in zip(test_src, predictions):
    inp = (s - 3).tolist()
    pred = (p[:SEQ_LEN] - 3).tolist()
    status = "correct" if pred == inp[::-1] else "wrong"
    print(f"input {inp}  ->  model {pred}   [{status}]")

big_src, _, big_tgt = make_batch(500)
pred = generate(model, big_src)
accuracy = (pred == big_tgt).all(dim=1).float().mean().item()
print(f"\nAccuracy on 500 new sequences: {accuracy:.1%}")

### Look inside: what did the attention learn?

To reverse a sequence, output position 1 must copy input position 8, output position 2 must copy input position 7, and so on.

The **cross-attention** heatmap below shows where the decoder looked in the input when writing each output token. You should see a clear **diagonal from top-right to bottom-left**. Nobody programmed this rule; the model **learned** it by adjusting its Q, K, V weights.

In [ ]:
sample = make_batch(1)[0]
generate(model, sample)       # run once so attention weights are stored
model.eval()
with torch.no_grad():
    enc_out = model.encode(sample)
    full_out = torch.cat([torch.tensor([[BOS]]), generate(model, sample)], dim=1)
    model.decode(full_out[:, :-1], enc_out)

in_labels  = [str(t - 3) for t in sample[0].tolist()]
out_labels = [str(t - 3) if t > 2 else "EOS" for t in full_out[0, 1:].tolist()]

cross = model.decoder[-1].cross_attn.attn_weights[0].mean(dim=0)    # average over heads
show_heatmap(cross, in_labels, out_labels, "Cross-attention (trained): decoder looking at the encoder",
             xlabel="Input digit (encoder)", ylabel="Output digit being written (decoder)")

fig, axes = plt.subplots(1, 4, figsize=(14, 3.5))
for h, ax in enumerate(axes):
    ax.imshow(model.decoder[-1].cross_attn.attn_weights[0, h].numpy(), cmap="viridis")
    ax.set_title(f"Head {h + 1}"); ax.set_xticks([]); ax.set_yticks([])
plt.suptitle("Each head can learn a slightly different pattern"); plt.tight_layout(); plt.show()

In [ ]:
with torch.no_grad():
    model.encode(sample)
enc_self = model.encoder[0].self_attn.attn_weights[0].mean(dim=0)
show_heatmap(enc_self, in_labels, in_labels, "Encoder self-attention (trained): input digits looking at each other")

## Part 11: Real Pre-trained Transformers with Hugging Face

Everything above is what happens *inside* real models. In practice, you load models that are already trained on huge amounts of text. **This part needs internet** (it downloads models, ~500 MB in total).

- **BERT** = encoder only → understands text
- **GPT-2** = decoder only → generates text

In [ ]:
!pip install -q transformers bertviz

### 11a. A real tokenizer (word pieces)

In [ ]:
from transformers import AutoTokenizer

tok = AutoTokenizer.from_pretrained("bert-base-uncased")
text = "The cat sat on the mat because it was exhausted"
print("Tokens:", tok.tokenize(text))
print("IDs   :", tok(text)["input_ids"])
print("Vocabulary size:", tok.vocab_size)

### 11b. Encoder (BERT): understanding context by filling in a blank

In [ ]:
from transformers import pipeline

fill = pipeline("fill-mask", model="bert-base-uncased")
for r in fill("The cat sat on the [MASK] because it was tired."):
    print(f"{r['token_str']:>10}  {r['score']:.2%}")

### 11c. Decoder (GPT-2): generating text one token at a time

In [ ]:
gen = pipeline("text-generation", model="gpt2")
print(gen("Transformers are used in", max_new_tokens=30, do_sample=True, temperature=0.8)[0]["generated_text"])

### 11d. See real attention with BertViz

Hover over a word on the left to see which words it attends to. Try **"it"** in different layers and heads: some heads link it to **"cat"**.

In [ ]:
from transformers import AutoModel
from bertviz import head_view

bert = AutoModel.from_pretrained("bert-base-uncased", output_attentions=True)
inputs = tok("The cat sat on the mat because it was tired", return_tensors="pt")
attention = bert(**inputs).attentions
head_view(attention, tok.convert_ids_to_tokens(inputs["input_ids"][0]))

## Summary: what you built

| Part | Component | What it does |
|---|---|---|
| 1 | Tokenization | Text → token IDs |
| 2 | Embedding | Token IDs → meaning vectors |
| 3 | Positional encoding | Adds word order |
| 4 | Self-attention | Every word looks at every other word (Q, K, V) |
| 5 | Multi-head attention | Several attention patterns in parallel |
| 6 | Masking | Decoder can't see future words |
| 7 | Feed-forward + Add & Norm | Process each word, keep training stable |
| 8 | Decoder layer + cross-attention | Decoder reads the encoder's output |
| 9 | Full Transformer | Encoder stack + decoder stack + output layer |
| 10 | Training + generation | Learns Q/K/V weights; generates token by token |
| 11 | Hugging Face | Same architecture, pre-trained at huge scale |

**Exercises**
1. In Part 10, change `num_heads` to 1 or `num_layers` to 1. Does it still learn?
2. Change the task to *sort* the digits: replace `digits.flip(dims=[1])` with `digits.sort(dim=1).values`. How does the cross-attention heatmap change?
3. Remove the causal mask in `decode` (pass `tgt_mask=None`). Training loss drops fast, but does generation still work? Why not?
4. Remove positional encoding (`+ self.pe[...]`). Can the model still reverse the sequence?